In [2]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim

1) bit conversion helpers

In [3]:
def int_to_bits(n, width):
    # LSB -> MSB
    return [(n >> i) & 1 for i in range(width)]

def bits_to_int(bits):
    # bits: [b0, b1, ...] (LSB -> MSB)
    value = 0
    for i, b in enumerate(bits):
        value |= (int(b) << i)
    return value

2) dataset: all 4-bit addition pairs
    input : 8 bits  = a[3:0] + b[3:0]
    output: 5 bits  = (a+b)[4:0]

In [4]:
def build_dataset():
    X = []
    Y = []

    for a in range(16):
        for b in range(16):
            x_bits = int_to_bits(a, 4) + int_to_bits(b, 4)
            y_bits = int_to_bits(a + b, 5)

            X.append(x_bits)
            Y.append(y_bits)

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    return X, Y

3) QAT-ready model

In [14]:
class AddMLP_QAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.fc1 = nn.Linear(8, 32, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 5, bias=False)
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.dequant(x)
        return x

4) backend selection

In [15]:
def choose_backend():
    engines = list(torch.backends.quantized.supported_engines)

    if "x86" in engines:
        return "x86"
    if "fbgemm" in engines:
        return "fbgemm"
    if "qnnpack" in engines:
        return "qnnpack"
    if "onednn" in engines:
        return "onednn"

    raise RuntimeError(f"No usable quantized backend found. supported_engines={engines}")

5) accuracy helpers
    We train with BCEWithLogitsLoss,
    so logit >= 0 means predicted bit = 1

In [16]:
@torch.no_grad()
def evaluate_from_logits(logits, y_true):
    pred_bits = (logits >= 0).float()
    bit_acc = (pred_bits == y_true).float().mean().item()
    sample_acc = (pred_bits == y_true).all(dim=1).float().mean().item()
    return bit_acc, sample_acc, pred_bits

@torch.no_grad()
def evaluate_model(model, X, Y):
    model.eval()
    logits = model(X)
    return evaluate_from_logits(logits, Y)

6) prepare QAT model

In [17]:
def make_qat_model():
    backend = choose_backend()
    torch.backends.quantized.engine = backend

    model = AddMLP_QAT().train()
    model.qconfig = torch.ao.quantization.get_default_qat_qconfig(backend)
    model = torch.ao.quantization.prepare_qat(model, inplace=False)
    return model, backend

7) train QAT

In [18]:
def train_qat(num_epochs=3000, lr=0.01, seed=0):
    torch.manual_seed(seed)

    X, Y = build_dataset()
    model, backend = make_qat_model()

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        model.train()

        logits = model(X)
        loss = criterion(logits, Y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0 or epoch == 1:
            bit_acc, sample_acc, _ = evaluate_model(model, X, Y)
            print(
                f"Epoch {epoch:4d} | "
                f"Loss {loss.item():.6f} | "
                f"Bit Acc {bit_acc*100:.2f}% | "
                f"Sample Acc {sample_acc*100:.2f}%"
            )

        # tiny trick often used in QAT:
        # later in training, stop updating observer stats
        if epoch == int(num_epochs * 0.7):
            model.apply(torch.ao.quantization.disable_observer)

        if epoch == int(num_epochs * 0.85):
            model.apply(torch.ao.quantization.disable_fake_quant)

        # if perfect already, stop early
        bit_acc, sample_acc, _ = evaluate_model(model, X, Y)
        if sample_acc == 1.0:
            print(f"\nReached 100% sample accuracy at epoch {epoch}.")
            break

    return model, X, Y, backend

8) convert to real INT8 model

In [19]:
def convert_to_int8(model_qat):
    model_int8 = copy.deepcopy(model_qat).cpu().eval()
    model_int8 = torch.ao.quantization.convert(model_int8, inplace=False)
    return model_int8

9) example inference

In [20]:
@torch.no_grad()
def test_examples(model, title="model"):
    tests = [(0, 0), (1, 2), (3, 5), (7, 8), (9, 6), (15, 15)]

    print(f"\nExample predictions ({title}):")
    for a, b in tests:
        x = torch.tensor([int_to_bits(a, 4) + int_to_bits(b, 4)], dtype=torch.float32)
        logits = model(x)
        pred_bits = (logits >= 0).int().squeeze(0).tolist()
        pred_value = bits_to_int(pred_bits)

        print(
            f"{a:2d} + {b:2d} = pred {pred_value:2d}, "
            f"bits {pred_bits}, gt {a+b:2d}"
        )

10) dump quantized params to text

In [ ]:
def _scalar_to_python(x):
    if isinstance(x, torch.Tensor):
        if x.numel() == 1:
            return x.item()
        return x.tolist()
    return x

def dump_quantized_tensor_info(f, name, qt):
    f.write(f"[{name}]\n")
    f.write(f"dtype = {qt.dtype}\n")
    f.write(f"shape = {tuple(qt.shape)}\n")
    f.write(f"qscheme = {qt.qscheme()}\n")

    if qt.qscheme() in (torch.per_tensor_affine, torch.per_tensor_symmetric):
        f.write(f"scale = {float(qt.q_scale())}\n")
        f.write(f"zero_point = {int(qt.q_zero_point())}\n")
    else:
        f.write(f"axis = {int(qt.q_per_channel_axis())}\n")
        f.write(f"scales = {qt.q_per_channel_scales().tolist()}\n")
        f.write(f"zero_points = {qt.q_per_channel_zero_points().tolist()}\n")

    f.write(f"int_repr = {qt.int_repr().tolist()}\n")
    f.write(f"dequantized = {qt.dequantize().tolist()}\n")
    f.write("\n")

def _get_bias_tensor(mod):
    b = getattr(mod, "bias", None)
    if callable(b):
        b = b()
    return b

def dump_int8_model_to_text(model_int8, path="mlp_qat_int8_params.txt"):
    with open(path, "w", encoding="utf-8") as f:
        f.write("=== Quantized MLP parameters ===\n")
        f.write(f"quantized_engine = {torch.backends.quantized.engine}\n\n")

        # input quant
        f.write("[input_quant]\n")
        f.write(f"scale = {float(model_int8.quant.scale)}\n")
        f.write(f"zero_point = {int(model_int8.quant.zero_point)}\n\n")

        # fc1
        fc1_bias = _get_bias_tensor(model_int8.fc1)
        f.write("[fc1]\n")
        f.write(f"output_scale = {float(model_int8.fc1.scale)}\n")
        f.write(f"output_zero_point = {int(model_int8.fc1.zero_point)}\n")
        if fc1_bias is None:
            f.write("bias = None\n\n")
        else:
            f.write(f"bias = {fc1_bias.detach().cpu().tolist()}\n\n")
        dump_quantized_tensor_info(f, "fc1.weight", model_int8.fc1.weight())

        # fc2
        fc2_bias = _get_bias_tensor(model_int8.fc2)
        f.write("[fc2]\n")
        f.write(f"output_scale = {float(model_int8.fc2.scale)}\n")
        f.write(f"output_zero_point = {int(model_int8.fc2.zero_point)}\n")
        if fc2_bias is None:
            f.write("bias = None\n\n")
        else:
            f.write(f"bias = {fc2_bias.detach().cpu().tolist()}\n\n")
        dump_quantized_tensor_info(f, "fc2.weight", model_int8.fc2.weight())

    print(f"Saved quantized parameters to: {path}")

11) main

In [22]:
if __name__ == "__main__":
    model_qat, X, Y, backend = train_qat(num_epochs=3000, lr=0.01, seed=0)

    bit_acc_qat, sample_acc_qat, _ = evaluate_model(model_qat, X, Y)
    print(f"\nQAT model backend       : {backend}")
    print(f"QAT model bit accuracy  : {bit_acc_qat*100:.2f}%")
    print(f"QAT model sample acc    : {sample_acc_qat*100:.2f}%")

    test_examples(model_qat, title="QAT fake-quant model")

    model_int8 = convert_to_int8(model_qat)

    bit_acc_int8, sample_acc_int8, _ = evaluate_model(model_int8, X, Y)
    print(f"\nINT8 model bit accuracy : {bit_acc_int8*100:.2f}%")
    print(f"INT8 model sample acc   : {sample_acc_int8*100:.2f}%")

    test_examples(model_int8, title="real INT8 model")

    dump_int8_model_to_text(model_int8, "mlp_qat_int8_params.txt")

C:\Users\user\AppData\Local\Temp\ipykernel_44740\2891419901.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model = torch.ao.quantization.prepare_qat(model, inplace=False)


Epoch    1 | Loss 0.696235 | Bit Acc 51.17% | Sample Acc 2.34%
Epoch  100 | Loss 0.611886 | Bit Acc 72.19% | Sample Acc 22.66%
Epoch  200 | Loss 0.563654 | Bit Acc 75.16% | Sample Acc 23.83%
Epoch  300 | Loss 0.523051 | Bit Acc 77.89% | Sample Acc 29.30%
Epoch  400 | Loss 0.473117 | Bit Acc 80.08% | Sample Acc 32.81%
Epoch  500 | Loss 0.424428 | Bit Acc 84.14% | Sample Acc 42.97%
Epoch  600 | Loss 0.386597 | Bit Acc 85.31% | Sample Acc 47.66%
Epoch  700 | Loss 0.349557 | Bit Acc 86.41% | Sample Acc 50.78%
Epoch  800 | Loss 0.311248 | Bit Acc 88.44% | Sample Acc 55.47%
Epoch  900 | Loss 0.280001 | Bit Acc 89.38% | Sample Acc 58.98%
Epoch 1000 | Loss 0.256812 | Bit Acc 91.02% | Sample Acc 64.06%
Epoch 1100 | Loss 0.233719 | Bit Acc 91.33% | Sample Acc 63.67%
Epoch 1200 | Loss 0.217007 | Bit Acc 92.66% | Sample Acc 69.14%
Epoch 1300 | Loss 0.198232 | Bit Acc 93.36% | Sample Acc 71.88%
Epoch 1400 | Loss 0.184343 | Bit Acc 94.22% | Sample Acc 76.17%
Epoch 1500 | Loss 0.167242 | Bit Acc 95.0

C:\Users\user\AppData\Local\Temp\ipykernel_44740\108738792.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.ao.quantization.convert(model_int8, inplace=False)


AttributeError: 'NoneType' object has no attribute 'tolist'